# Hannah — real talking video (free test) · version 2

Free open-source models: **Kokoro-82M** (her voice, Apache-2.0) and **Ditto** (talking video, Apache-2.0).

**How to run (about 15–20 minutes):**
1. Menu **Runtime → Disconnect and delete runtime** (starts clean), then **Runtime → Change runtime type → T4 GPU → Save**
2. Menu **Runtime → Run all**
3. When step 3 shows **Choose files**, pick Hannah's picture from **Desktop → Hannah video test**
4. At the end `hannah_test.zip` downloads. Send it back.

Every step checks itself and prints **OK** or a clear problem.

In [ ]:
# 1. Check the free GPU
import subprocess, sys, os
os.environ['MPLBACKEND'] = 'Agg'   # the tools run in their own Python; Colab's chart setting would break them
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or 'NO GPU! Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.')
print('Colab Python:', sys.version.split()[0])

In [ ]:
# 2. Install everything in its own clean Python 3.10 (5–8 minutes)
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null 2>&1
!rm -rf /content/ditto && git clone -q --depth 1 https://github.com/antgroup/ditto-talkinghead /content/ditto
!pip install -q uv
!uv venv -q --seed --python 3.10 /content/venv
!uv pip install -q --python /content/venv/bin/python torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /content/venv/bin/python librosa filetype imageio imageio-ffmpeg scikit-image scipy cython setuptools einops tqdm opencv-python-headless mediapipe onnxruntime-gpu "kokoro>=0.9.4" "transformers<4.50" soundfile huggingface_hub
!/content/venv/bin/python -m spacy download en_core_web_sm -q > /dev/null 2>&1
!/content/venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('digital-avatar/ditto-talkinghead', local_dir='/content/ditto/checkpoints', allow_patterns=['ditto_pytorch/*', 'ditto_cfg/*'])"
# check that every piece is really there
!/content/venv/bin/python -c "import torch, filetype, librosa, cv2, mediapipe, onnxruntime, skimage, kokoro, soundfile, einops, pyximport; print('GPU for torch:', torch.cuda.is_available()); print('INSTALL OK')"
!ls /content/ditto/checkpoints/ditto_pytorch /content/ditto/checkpoints/ditto_cfg

In [ ]:
# 3. Upload Hannah's picture
from google.colab import files
import shutil, os
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, '/content/hannah.jpg')
print('Picture ready:', name)

In [ ]:
%%writefile /content/tts.py
# 4. Her voice: text → speech with Kokoro
import sys, numpy as np, soundfile as sf
from kokoro import KPipeline
text, voice, out = sys.argv[1], sys.argv[2], sys.argv[3]
pipe = KPipeline(lang_code='b' if voice.startswith('b') else 'a')
audio = np.concatenate([np.asarray(a) for _, _, a in pipe(text, voice=voice, speed=0.95)])
pad = np.zeros(int(24000 * 0.35), dtype=np.float32)          # calm start and end, for joining clips
sf.write(out, np.concatenate([pad, audio, pad]), 24000)
print('wrote', out)

In [ ]:
# 4b. The greeting, 3 voice samples to choose from, and silence for the idle video
import os, numpy as np, soundfile as sf
os.makedirs('/content/out', exist_ok=True)
GREETING = "Hello and welcome to Nexora! I'm Hannah, your virtual receptionist. How can I help you today?"
!/content/venv/bin/python /content/tts.py "$GREETING" af_heart /content/out/greeting.wav
for v in ['af_heart', 'af_bella', 'bf_emma']:
    !/content/venv/bin/python /content/tts.py "Hi, I'm Hannah. Welcome to Nexora." {v} /content/out/voice_sample_{v}.wav
sf.write('/content/out/silence.wav', np.zeros(16000 * 6, dtype=np.float32), 16000)
print('VOICE OK' if os.path.exists('/content/out/greeting.wav') else 'VOICE FAILED: send a screenshot of the red text above')

In [ ]:
# 5. Make the videos with Ditto (5–10 minutes)
%cd /content/ditto
!/content/venv/bin/python inference.py --data_root ./checkpoints/ditto_pytorch --cfg_pkl ./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl --audio_path /content/out/greeting.wav --source_path /content/hannah.jpg --output_path /content/out/hannah_greeting.mp4
!/content/venv/bin/python inference.py --data_root ./checkpoints/ditto_pytorch --cfg_pkl ./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl --audio_path /content/out/silence.wav --source_path /content/hannah.jpg --output_path /content/out/hannah_idle.mp4
!ls -la /content/out
print('VIDEO OK' if os.path.exists('/content/out/hannah_greeting.mp4') else 'VIDEO FAILED: send a screenshot of the red text above')

In [ ]:
# 6. Watch it here, then download everything
from IPython.display import HTML, display
from base64 import b64encode
if os.path.exists('/content/out/hannah_greeting.mp4'):
    mp4 = open('/content/out/hannah_greeting.mp4', 'rb').read()
    display(HTML('<video width=720 controls autoplay><source src="data:video/mp4;base64,' + b64encode(mp4).decode() + '"></video>'))
!cd /content/out && zip -q -r /content/hannah_test.zip .
files.download('/content/hannah_test.zip')